# ARC-AGI-3 Duck v12 — Qwen3.8 + ADLDB DWE

Comparison target: `wellkilo/arc3lab-duck-v12-control-seed-20260819`

Default seed: `20260819`

One real trajectory/game; Qwen3.8-27B-FP8; 32K working context; full-frame request; ACTION7; state graph off; post-move ADL; statistical no-impact; game/strategy exploit weights; dynamic budget/change-policy/stop-loss; deterministic per-move logs.


In [ ]:
import json
import os
import pickle
import random
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()
DEFAULT_CONTROL_SEED = 20260819
CONTROL_SEED = int(os.environ.get("ADLDB_CONTROL_SEED", str(DEFAULT_CONTROL_SEED)))
KNOWN_PUBLIC_CONTROL_SEEDS = (20260819, 20260807)
ANALYZER_MODEL_ID = "Qwen/Qwen3.8-27B-FP8"
ANALYZER_CONTEXT_WINDOW = 32768
VLLM_MAX_MODEL_LEN = 65536

os.environ["PYTHONHASHSEED"] = str(CONTROL_SEED)
os.environ["ADLDB_CONTROL_SEED"] = str(CONTROL_SEED)
os.environ["TAAF_CONTROL_SEED"] = str(CONTROL_SEED)
os.environ["VLLM_SEED"] = str(CONTROL_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(CONTROL_SEED)
try:
    import numpy as _np
    _np.random.seed(CONTROL_SEED)
except Exception:
    _np = None
try:
    import torch as _torch
    _torch.manual_seed(CONTROL_SEED)
    if _torch.cuda.is_available():
        _torch.cuda.manual_seed_all(CONTROL_SEED)
except Exception:
    _torch = None

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["INFERENCE_ANALYZER_MODEL"] = ANALYZER_MODEL_ID
os.environ["LOCAL_ANALYZER_MODEL_ID"] = ANALYZER_MODEL_ID
os.environ.setdefault("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1")
os.environ["TAAF_MAX_OUTPUT_TOKENS"] = "8192"
os.environ["TAAF_TOOL_STEPS"] = "8"
os.environ["TAAF_TEMPERATURE"] = "1.0"
os.environ["TAAF_TOP_P"] = "0.95"
os.environ["TAAF_TOP_K"] = "20"
os.environ["TAAF_CONTEXT_WINDOW"] = str(ANALYZER_CONTEXT_WINDOW)
os.environ["ARC3_FRAME_MODE"] = "full"
os.environ["ARC3_STATE_GRAPH"] = "off"
os.environ["ARC3_REEXPLORE_STRICT"] = "0"
os.environ["ADLDB_NO_IMPACT"] = "on"

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)
WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(
    "ADLDB CONTROL "
    f"seed={CONTROL_SEED} known_public_seeds={KNOWN_PUBLIC_CONTROL_SEEDS} "
    f"model={ANALYZER_MODEL_ID} context={ANALYZER_CONTEXT_WINDOW} "
    f"frame_mode={os.environ['ARC3_FRAME_MODE']} state_graph={os.environ['ARC3_STATE_GRAPH']} "
    f"TRUE_SUBMISSION={TRUE_SUBMISSION}",
    flush=True,
)


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# === ALL REQUIRED INPUTS — RESOLVE BEFORE INSTALL/INFERENCE ===
REQUIRED_COMPETITION = "arc-prize-2026-arc-agi-3"
TAAF_SOURCE_REF = "jeroencottaar/taaf-kaggle-source-share"
VLLM_WHEELHOUSE_REF = "driessmit1/arc3-vllm-h100-wheelhouse-v3"
LEGACY_TAAF_MODEL_REF = "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"
PREFERRED_QWEN38_REF = "driessmit1/vrfai-qwen3-8-27b-fp8-hf-snapshot"
DATASET_SOURCES = [TAAF_SOURCE_REF, VLLM_WHEELHOUSE_REF]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [KAGGLE_INPUT_ROOT / slug, KAGGLE_INPUT_ROOT / "datasets" / owner / slug]


def _competition_mount_candidates(slug: str) -> list[Path]:
    return [KAGGLE_INPUT_ROOT / slug, KAGGLE_INPUT_ROOT / "competitions" / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(paths):
    return next((p for p in paths if p.exists()), None)


def _find_named(root: Path, name: str):
    if not root.exists():
        return None
    direct = root / name
    if direct.exists():
        return direct
    try:
        return next(root.rglob(name), None)
    except OSError:
        return None


def _require(path, label):
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"REQUIRED INPUT MISSING: {label}. Attach it before the scored run.")
    return Path(path).resolve()


def _resolve_dataset(ref, marker=None):
    root = _first_existing(_dataset_mount_candidates(ref))
    if root is not None and (marker is None or _find_named(root, marker) is not None):
        return root.resolve()
    if marker and KAGGLE_INPUT_ROOT.exists():
        hit = _find_named(KAGGLE_INPUT_ROOT, marker)
        if hit is not None:
            return hit.parent.resolve()
    raise FileNotFoundError(f"Missing dataset {ref}")


def _has_weights(root):
    return (
        (root / "model.safetensors").exists()
        or (root / "model.safetensors.index.json").exists()
        or any(root.glob("*.safetensors"))
    )


def _qwen38_score(model_dir, cfg):
    text = (str(model_dir) + " " + json.dumps(cfg, sort_keys=True, default=str)).lower()
    score = 0
    for token, points in (("qwen3.8", 140), ("qwen3-8", 130), ("qwen3_8", 130), ("qwen38", 120), ("27b", 25), ("fp8", 25)):
        if token in text:
            score += points
    if str(cfg.get("model_type", "")).lower() == "qwen3_5":
        score += 10
    text_cfg = cfg.get("text_config", {})
    if isinstance(text_cfg, dict):
        score += 5 if int(text_cfg.get("hidden_size", 0) or 0) == 5120 else 0
        score += 5 if int(text_cfg.get("num_hidden_layers", 0) or 0) == 64 else 0
    qcfg = json.dumps(cfg.get("quantization_config", {}), sort_keys=True).lower()
    score += 10 if ("fp8" in qcfg or "float8" in qcfg) else 0
    return score


def _resolve_qwen38():
    explicit = os.environ.get("ADLDB_QWEN38_MODEL_DIR", "").strip()
    if explicit:
        root = Path(explicit)
        _require(root / "config.json", "Qwen3.8 config.json")
        if not _has_weights(root):
            raise FileNotFoundError(f"No model weights in {root}")
        return root.resolve(), "explicit-env"
    candidates = []
    roots = [p for p in _dataset_mount_candidates(PREFERRED_QWEN38_REF) if p.exists()]
    if KAGGLE_INPUT_ROOT.exists():
        roots.append(KAGGLE_INPUT_ROOT)
    seen = set()
    for root in roots:
        try:
            cfg_paths = list(root.rglob("config.json"))
        except OSError:
            cfg_paths = []
        for cfg_path in cfg_paths:
            model_dir = cfg_path.parent
            if model_dir in seen or not _has_weights(model_dir):
                continue
            seen.add(model_dir)
            try:
                cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            score = _qwen38_score(model_dir, cfg)
            if score >= 120:
                candidates.append((score, model_dir))
    if not candidates:
        raise FileNotFoundError(
            "Qwen3.8-27B-FP8 is not mounted. Attach the complete "
            f"{ANALYZER_MODEL_ID} Kaggle/Hugging Face snapshot. "
            "The notebook discovers its mount path by model contents."
        )
    candidates.sort(key=lambda row: (row[0], -len(str(row[1]))), reverse=True)
    score, root = candidates[0]
    return root.resolve(), f"content-discovery(score={score})"


ARC_COMPETITION_ROOT = _first_existing(_competition_mount_candidates(REQUIRED_COMPETITION))
if ARC_COMPETITION_ROOT is None and KAGGLE_INPUT_ROOT.exists():
    wheels = _find_named(KAGGLE_INPUT_ROOT, "arc_agi_3_wheels")
    if wheels is not None and wheels.is_dir():
        ARC_COMPETITION_ROOT = wheels.parent
ARC_COMPETITION_ROOT = _require(ARC_COMPETITION_ROOT, f"competition:{REQUIRED_COMPETITION}")
ARC_WHEELS_DIR = ARC_COMPETITION_ROOT / "arc_agi_3_wheels"
if not ARC_WHEELS_DIR.is_dir():
    ARC_WHEELS_DIR = _require(_find_named(ARC_COMPETITION_ROOT, "arc_agi_3_wheels"), "arc_agi_3_wheels")
ARC_WHEELS_DIR = ARC_WHEELS_DIR.resolve()
ARC_ENVIRONMENTS_DIR = (ARC_COMPETITION_ROOT / "environment_files").resolve()
if not TRUE_SUBMISSION:
    ARC_ENVIRONMENTS_DIR = _require(ARC_ENVIRONMENTS_DIR if ARC_ENVIRONMENTS_DIR.is_dir() else None, "environment_files")

bundle_mount = _resolve_dataset(TAAF_SOURCE_REF, DATASET_BUNDLE_MARKER)
bundle_marker = _find_named(bundle_mount, DATASET_BUNDLE_MARKER)
BUNDLE_DIR = _require(bundle_marker.parent if bundle_marker else None, "TAAF source bundle")
for required in ("src", "setup_commands.json", "teardown_commands.json", "deploy_target.pkl", "benchmark_initial.pkl"):
    _require(BUNDLE_DIR / required, f"TAAF component:{required}")

wheel_mount = _resolve_dataset(VLLM_WHEELHOUSE_REF, "requirements.lock")
wheel_lock = _find_named(wheel_mount, "requirements.lock")
VLLM_WHEELHOUSE_DIR = _require(wheel_lock.parent if wheel_lock else None, "vLLM requirements.lock")
QWEN_MODEL_DIR, QWEN_MODEL_SOURCE = _resolve_qwen38()

kaggle_input_paths = {
    TAAF_SOURCE_REF: str(BUNDLE_DIR),
    VLLM_WHEELHOUSE_REF: str(VLLM_WHEELHOUSE_DIR),
    LEGACY_TAAF_MODEL_REF: str(QWEN_MODEL_DIR),
    PREFERRED_QWEN38_REF: str(QWEN_MODEL_DIR),
    ANALYZER_MODEL_ID: str(QWEN_MODEL_DIR),
    f"competition:{REQUIRED_COMPETITION}": str(ARC_COMPETITION_ROOT),
}
setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_KAGGLE_BUNDLE_DIR": str(BUNDLE_DIR),
    "ARC_AGI3_COMPETITION_ROOT": str(ARC_COMPETITION_ROOT),
    "ARC_AGI3_WHEELS_DIR": str(ARC_WHEELS_DIR),
    "ARC_AGI3_ENVIRONMENTS_DIR": str(ARC_ENVIRONMENTS_DIR),
    "TAAF_VLLM_WHEELHOUSE": str(VLLM_WHEELHOUSE_DIR),
    "TAAF_QWEN_MODEL_DIR": str(QWEN_MODEL_DIR),
    "ADLDB_QWEN38_MODEL_DIR": str(QWEN_MODEL_DIR),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n", encoding="utf-8")
manifest = {
    "competition": str(ARC_COMPETITION_ROOT),
    "arc_wheels": str(ARC_WHEELS_DIR),
    "environment_files": str(ARC_ENVIRONMENTS_DIR),
    "taaf_source": str(BUNDLE_DIR),
    "vllm_wheelhouse": str(VLLM_WHEELHOUSE_DIR),
    "qwen38_model": str(QWEN_MODEL_DIR),
    "qwen38_resolution": QWEN_MODEL_SOURCE,
    "model_id": ANALYZER_MODEL_ID,
    "control_seed": CONTROL_SEED,
    "known_public_seeds": list(KNOWN_PUBLIC_CONTROL_SEEDS),
}
(WORKING_DIR / "adldb_input_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("=" * 92)
print("ADLDB REQUIRED INPUTS — ALL RESOLVED")
for key, value in manifest.items():
    print(f"{key:22s} = {value}")
print("=" * 92)


In [ ]:
# === INSTALL ARC RUNTIME FROM THE RESOLVED COMPETITION INPUT ===
# No internet is required.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        str(ARC_WHEELS_DIR),
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)
print(f"taaf.kaggle: arc-agi installed from {ARC_WHEELS_DIR}")


## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# === PUBLISH RESOLVED INPUTS TO TAAF SETUP ===
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])
setup_env.update({
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_DIR": str(QWEN_MODEL_DIR),
    "ADLDB_QWEN38_MODEL_DIR": str(QWEN_MODEL_DIR),
})
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")
print(f"taaf.kaggle: Qwen3.8 path = {QWEN_MODEL_DIR}")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands from the immutable source bundle.
def _patch_setup_for_qwen38(command: str) -> str:
    patched = str(command)
    patched = patched.replace("SERVED_MODEL_NAME = 'vrfai/Qwen3.6-27B-FP8'", "SERVED_MODEL_NAME = 'Qwen/Qwen3.8-27B-FP8'")
    patched = patched.replace("'{\"preserve_thinking\": true}'", "'{\"enable_thinking\": true, \"preserve_thinking\": true}'")
    patched = patched.replace("'LOCAL_ANALYZER_TEMPERATURE': '0.6'", "'LOCAL_ANALYZER_TEMPERATURE': '1.0'")
    seed_needle = "        '--max-model-len',\n        str(VLLM_MAX_MODEL_LEN),"
    seed_replacement = "        '--seed',\n        os.environ.get('ADLDB_CONTROL_SEED', '20260819'),\n        '--max-model-len',\n        str(VLLM_MAX_MODEL_LEN),"
    if "'--seed'" not in patched and '"--seed"' not in patched:
        patched = patched.replace(seed_needle, seed_replacement)
    return patched

env = _command_env()
for raw_command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    command = _patch_setup_for_qwen38(raw_command)
    print(f"taaf.kaggle: setup model={ANALYZER_MODEL_ID} seed={CONTROL_SEED}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    env = _command_env(); os.environ.update(env)
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Bound generation before inference modules are imported.
# The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '1.0'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [ ]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only current-game observations, actions, rewards, transitions, Qwen's pass-1 transcript, and Gemma's review of that transcript may cross into pass 2.


In [ ]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "same_current_run_same_game_post_move_adl",
        "same_current_run_same_game_dual_path_decisions",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
        "prior_submission_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict current-game/current-run no-prior contract active")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


## 6. ADLDB + Difference-Weighted Exploitation configuration

Every discovered game is still run once. `LS20_MAX_MOVES` remains the absolute safety ceiling, while DWE computes a **live per-game budget** from current-game evidence. A successful transition gets a protected exploit window; repeated no-progress, stalls, and loops reduce strategy weight and can trigger policy change or stop-loss.


In [ ]:
# === ADLDB / DWE v3 CONFIGURATION ===
STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = 4
TARGET_SCORE_GAMES = 10
TARGET_MIN_GAME_SCORE = 0.40
LS20_MAX_MOVES = 309
MAX_STALL_ACTIONS = 12
MAX_NO_PROGRESS_ACTIONS = 24
MAX_NO_IMPACT_ACTIONS = 4
MIN_OBSERVATION_ACTIONS = 8
SUCCESS_PROTECT_ACTIONS = 18
DWE_STRICT_LOG_COVERAGE = True
NO_IMPACT_BAND_WINDOW = 20
NO_IMPACT_BAND_WARMUP = 8
NO_IMPACT_BAND_THRESHOLD = 0.90
EXPLOIT_WEIGHTS = {
    "score": 1.50,
    "level_complete": 4.00,
    "progress_velocity": 2.25,
    "novel_state": 1.00,
    "causal_confidence": 1.75,
    "target_proximity": 2.50,
    "stall": -2.00,
    "repeat_loop": -3.50,
    "no_progress": -2.75,
    "no_impact": -4.25,
    "terminal_loss": -4.00,
}
GAME_WEIGHT_DECAY = 0.94
STRATEGY_WEIGHT_DECAY = 0.88
GAME_WEIGHT_LIMIT = 12.0
STRATEGY_WEIGHT_LIMIT = 12.0
DWE_BUDGET_TIERS = ((6.0,309),(3.0,260),(1.0,210),(-1.0,160),(-3.0,120),(-999.0,84))
_original_game_budget = float(getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0)
_original_concurrency = max(1, int(getattr(bm.solver, "concurrency", 1) or 1))
bm.solver.concurrency = TARGET_CONCURRENCY
bm.solver.max_actions_per_game = LS20_MAX_MOVES
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None
_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)
assert bm.solver.concurrency == TARGET_CONCURRENCY
assert bm.solver.max_actions_per_game == LS20_MAX_MOVES
print(
    "REAL RUN CONFIG "
    f"model={ANALYZER_MODEL_ID} seed={CONTROL_SEED} concurrency={TARGET_CONCURRENCY} "
    f"hard_ceiling={LS20_MAX_MOVES} stall={MAX_STALL_ACTIONS} no_progress={MAX_NO_PROGRESS_ACTIONS} "
    f"no_impact={MAX_NO_IMPACT_ACTIONS} band={NO_IMPACT_BAND_WINDOW}/{NO_IMPACT_BAND_WARMUP}/{NO_IMPACT_BAND_THRESHOLD:.2f} "
    f"context={_graft_flags['context_window']} frame_mode={os.environ.get('ARC3_FRAME_MODE')} state_graph={os.environ.get('ARC3_STATE_GRAPH')}",
    flush=True,
)
print("DWE WEIGHTS:", json.dumps(EXPLOIT_WEIGHTS, sort_keys=True), flush=True)


## 7. Closed-loop ADL + Difference-Weighted Exploitation

The model still performs the two-plan `EXPLOIT` vs `EXPLORE` comparison before each action and a post-move ADL update afterward. In addition, this cell installs a **deterministic runtime DWE auditor** at the real `GameAPI` action boundary.

For every committed action the notebook prints:

- `DWE PRE`: prior game/strategy weights, current decision, live budget, stall/no-progress counters.
- `DWE POST`: before/after score and levels, reward, board/state-change evidence, novelty/loop signals, each weighted term, new weights, and the next allocator decision.

The same records are written to `/kaggle/working/dwe_move_events.jsonl`.


In [ ]:
# === CLOSED-LOOP ADL + DIFFERENCE-WEIGHTED EXPLOITATION ===
import asyncio
import contextvars
import hashlib
import inspect
import json
import math
import os
import sys
import threading
from collections import deque
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Mapping
from inference.agent.tool_agent import ToolAgent

DUAL_PATH_ENABLED = True
POST_MOVE_ADL_ENABLED = True
DUAL_PATH_CANDIDATES = 2

DUAL_PATH_POLICY_LOG = WORKING_DIR / "dual_path_policy_events.jsonl"
POST_MOVE_ADL_LOG = WORKING_DIR / "post_move_adl_events.jsonl"
DWE_MOVE_LOG = WORKING_DIR / "dwe_move_events.jsonl"
DWE_SUMMARY_LOG = WORKING_DIR / "dwe_game_summaries.jsonl"

for _path in (DUAL_PATH_POLICY_LOG, POST_MOVE_ADL_LOG, DWE_MOVE_LOG, DWE_SUMMARY_LOG):
    try:
        _path.unlink(missing_ok=True)
    except Exception:
        pass

CLOSED_LOOP_ADL_INSTRUCTION = r"""
ADLDB CURRENT-GAME POLICY
Use one real trajectory and no cross-game priors.
Before each move compare A) exploit verified current-game causality and B) explore the highest-information legal alternative.
Prefer level completion/depth over cosmetic motion. Treat deterministic HUD/timer changes as housekeeping.
After each move obey DWE: EXPLOIT/HARD_EXPLOIT=reuse verified causality; CHANGE_POLICY=stop repeating local behavior; STOP_LOSS=stop spending actions on the failed trajectory.
ACTION7 is game-specific when legal. Inspect animation only when it resolves causality.
""".strip()


class ClosedLoopADLToolAgent(ToolAgent):
    """Duck ToolAgent with dual-path ADL and explicit DWE reasoning requirements."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._system_prompt += "\n\n" + CLOSED_LOOP_ADL_INSTRUCTION


def _closed_loop_adl_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or ANALYZER_MODEL_ID
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    return ClosedLoopADLToolAgent(
        model=model,
        timeout=bm.solver.analyzer_timeout,
        save_request_logs=bm.solver.save_request_logs,
        base_url=base_url,
        provider="vllm",
    )


bm.solver.analyzer_factory = _closed_loop_adl_analyzer_factory


def _clip(value, low, high):
    return max(low, min(high, float(value)))


def _num(value, default=None):
    if value is None or isinstance(value, bool):
        return default
    try:
        x = float(value)
        if math.isfinite(x):
            return x
    except Exception:
        pass
    return default


def _read(obj, names, default=None):
    if obj is None:
        return default
    for name in names:
        try:
            if isinstance(obj, Mapping) and name in obj:
                value = obj[name]
            elif hasattr(obj, name):
                value = getattr(obj, name)
            else:
                continue
            if callable(value):
                continue
            if value is not None:
                return value
        except Exception:
            continue
    return default


def _walk_candidates(obj, max_depth=2):
    """Yield a small, safe object graph for score/state field discovery."""
    seen = set()
    queue = deque([(obj, 0)])
    child_names = (
        "state", "game_state", "observation", "result", "info", "metadata",
        "response", "frame", "board", "env", "game", "run",
    )
    while queue:
        cur, depth = queue.popleft()
        if cur is None or id(cur) in seen:
            continue
        seen.add(id(cur))
        yield cur
        if depth >= max_depth:
            continue
        for name in child_names:
            nxt = _read(cur, (name,), None)
            if nxt is not None and not isinstance(nxt, (str, bytes, int, float, bool)):
                queue.append((nxt, depth + 1))


def _find_number(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _num(_read(candidate, names, None), None)
            if value is not None:
                return value
    return default


def _find_bool(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _read(candidate, names, None)
            if isinstance(value, bool):
                return value
            if isinstance(value, (int, float)) and value in (0, 1):
                return bool(value)
            if isinstance(value, str):
                v = value.strip().lower()
                if v in {"true", "yes", "won", "lost", "done", "terminal", "game_over"}:
                    return True
                if v in {"false", "no", "playing", "active", "running"}:
                    return False
    return default


def _extract_game_id(*objs):
    names = ("game_id", "env_name", "environment_id", "game_name", "name")
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=3):
            value = _read(candidate, names, None)
            if value is not None:
                text = str(value).strip()
                if text:
                    return text
    return "unknown"


def _extract_action(args, kwargs):
    for key in ("action", "action_name", "action_spec", "move", "command"):
        if key in kwargs:
            return str(kwargs[key])
    for value in args:
        if value is None:
            continue
        text = str(value)
        if text and len(text) <= 300:
            return text
    return "unknown"


def _stable_signature(*objs):
    """Hash only likely visual/state payloads; avoid timestamps/request metadata."""
    field_names = (
        "board", "grid", "frame", "image", "observation", "pixels",
        "screen", "state_matrix", "board_state",
    )
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=2):
            for name in field_names:
                value = _read(candidate, (name,), None)
                if value is None:
                    continue
                try:
                    if hasattr(value, "tolist"):
                        value = value.tolist()
                    payload = json.dumps(value, sort_keys=True, default=str, separators=(",", ":"))
                except Exception:
                    payload = repr(value)
                if payload and len(payload) > 4:
                    return hashlib.sha1(payload[:2_000_000].encode("utf-8", "replace")).hexdigest()[:16]
    return None


def _snapshot(api, result=None):
    objs = tuple(x for x in (result, api) if x is not None)
    score = _find_number(objs, ("score", "current_score", "total_score", "game_score", "final_score"), None)
    levels = _find_number(objs, ("levels_completed", "level_completed_count", "completed_levels", "level"), None)
    reward = _find_number((result,), ("reward", "score_delta", "delta_reward"), None)
    board_changed = _find_bool((result,), ("board_changed", "state_changed", "frame_changed", "changed"), None)
    level_completed = _find_bool((result,), ("level_completed", "completed_level", "level_won"), None)
    game_over = _find_bool(objs, ("game_over", "done", "terminal", "is_done", "finished"), None)
    won = _find_bool(objs, ("won", "is_won", "victory"), None)
    lost = _find_bool(objs, ("lost", "is_lost", "defeat"), None)
    signature = _stable_signature(result, api)
    return {
        "score": score,
        "levels": int(levels) if levels is not None else None,
        "reward": reward,
        "board_changed": board_changed,
        "level_completed": level_completed,
        "game_over": game_over,
        "won": won,
        "lost": lost,
        "signature": signature,
    }


@dataclass
class DWEGameState:
    game_id: str
    move: int = 0
    last_score: float = 0.0
    last_levels: int = 0
    game_weight: float = 0.0
    strategy_weight: float = 0.0
    combined_weight: float = 0.0
    progress_velocity: float = 0.0
    stall_streak: int = 0
    no_progress_streak: int = 0
    repeat_streak: int = 0
    success_protect_until: int = 0
    live_budget: int = 160
    decision: str = "EXPLORE"
    reason: str = "initial observation"
    last_signature: str | None = None
    seen_signatures: deque = field(default_factory=lambda: deque(maxlen=96))
    last_terms: dict = field(default_factory=dict)


class DifferenceWeightedExploitation:
    def __init__(self):
        self._states = {}
        self._lock = threading.RLock()

    def state(self, game_id):
        key = str(game_id or "unknown")
        with self._lock:
            if key not in self._states:
                self._states[key] = DWEGameState(game_id=key)
            return self._states[key]

    @staticmethod
    def _budget(weight):
        for threshold, budget in DWE_BUDGET_TIERS:
            if weight >= threshold:
                return int(min(LS20_MAX_MOVES, budget))
        return int(LS20_MAX_MOVES)

    def pre(self, game_id, action):
        with self._lock:
            st = self.state(game_id)
            print(
                "DWE PRE "
                f"game={st.game_id} move={st.move + 1:03d} action={action} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} mode={st.decision} "
                f"live_budget={st.live_budget}/{LS20_MAX_MOVES} "
                f"stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"protect_until={st.success_protect_until} reason={st.reason}",
                flush=True,
            )
            return {
                "move": st.move + 1,
                "score": st.last_score,
                "levels": st.last_levels,
                "signature": st.last_signature,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "decision": st.decision,
            }

    def post(self, game_id, action, before, after):
        with self._lock:
            st = self.state(game_id)
            st.move += 1

            before_score = _num(before.get("score"), st.last_score)
            if before_score is None:
                before_score = st.last_score
            after_score = _num(after.get("score"), None)
            reward = _num(after.get("reward"), 0.0) or 0.0
            if after_score is None:
                after_score = before_score + reward
            score_delta = after_score - before_score

            before_levels = before.get("levels")
            if before_levels is None:
                before_levels = st.last_levels
            after_levels = after.get("levels")
            level_event = bool(after.get("level_completed"))
            if after_levels is None:
                after_levels = before_levels + (1 if level_event else 0)
            level_delta = max(0, int(after_levels) - int(before_levels))
            level_event = bool(level_event or level_delta > 0)

            sig = after.get("signature")
            prev_sig = st.last_signature
            seen_before = set(st.seen_signatures)
            novel = bool(sig and sig not in seen_before)
            repeated = bool(sig and (sig == prev_sig or sig in seen_before))
            board_changed = after.get("board_changed")
            if board_changed is None and sig and prev_sig:
                board_changed = sig != prev_sig
            if board_changed is None:
                board_changed = bool(score_delta != 0 or level_event or reward != 0)

            positive_score = score_delta > 1e-9
            positive_reward = reward > 1e-9
            meaningful_progress = bool(level_event or positive_score or positive_reward)
            state_activity = bool(board_changed or novel)
            loop_signal = bool(repeated and not meaningful_progress)

            if meaningful_progress:
                st.no_progress_streak = 0
            else:
                st.no_progress_streak += 1

            if meaningful_progress or state_activity:
                st.stall_streak = 0
            else:
                st.stall_streak += 1

            if loop_signal:
                st.repeat_streak += 1
            else:
                st.repeat_streak = 0

            # Progress value is deliberately conservative: visual novelty alone is useful
            # information, but weaker than verified score/reward/level progress.
            progress_value = 0.0
            if level_event:
                progress_value += 1.00
            if positive_score:
                progress_value += min(0.65, 0.20 + abs(score_delta))
            if positive_reward:
                progress_value += min(0.40, 0.10 + abs(reward))
            if board_changed:
                progress_value += 0.12
            if novel:
                progress_value += 0.10
            if loop_signal:
                progress_value -= 0.30
            if not meaningful_progress and not state_activity:
                progress_value -= 0.15
            progress_value = _clip(progress_value, -1.0, 1.0)
            st.progress_velocity = _clip(0.70 * st.progress_velocity + 0.30 * progress_value, -1.0, 1.0)

            # Causal confidence is tied only to observed current-game consequences.
            causal_confidence = 0.0
            if level_event:
                causal_confidence = 1.0
            elif positive_score:
                causal_confidence = 0.85
            elif positive_reward:
                causal_confidence = 0.70
            elif board_changed and novel:
                causal_confidence = 0.35
            elif board_changed:
                causal_confidence = 0.20

            normalized_score = _clip(after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 2.0)
            target_proximity = 1.0 if after_score >= TARGET_MIN_GAME_SCORE else _clip(
                after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 1.0
            )
            # A stagnant near-target game should not monopolize budget, so proximity is
            # gated by recent causal progress velocity.
            target_activity_gate = 0.20 + 0.80 * max(0.0, st.progress_velocity)
            target_signal = target_proximity * target_activity_gate
            stall_ratio = _clip(st.stall_streak / max(MAX_STALL_ACTIONS, 1), 0.0, 1.0)
            no_progress_ratio = _clip(st.no_progress_streak / max(MAX_NO_PROGRESS_ACTIONS, 1), 0.0, 1.0)
            repeat_ratio = _clip(st.repeat_streak / 4.0, 0.0, 1.0)
            terminal_loss = bool(after.get("lost") or (after.get("game_over") and not after.get("won")))

            terms = {
                "score": EXPLOIT_WEIGHTS["score"] * normalized_score,
                "level_complete": EXPLOIT_WEIGHTS["level_complete"] * (1.0 if level_event else 0.0),
                "progress_velocity": EXPLOIT_WEIGHTS["progress_velocity"] * st.progress_velocity,
                "novel_state": EXPLOIT_WEIGHTS["novel_state"] * (1.0 if novel else 0.0),
                "causal_confidence": EXPLOIT_WEIGHTS["causal_confidence"] * causal_confidence,
                "target_proximity": EXPLOIT_WEIGHTS["target_proximity"] * target_signal,
                "stall": EXPLOIT_WEIGHTS["stall"] * stall_ratio,
                "repeat_loop": EXPLOIT_WEIGHTS["repeat_loop"] * repeat_ratio,
                "no_progress": EXPLOIT_WEIGHTS["no_progress"] * no_progress_ratio,
                "terminal_loss": EXPLOIT_WEIGHTS["terminal_loss"] * (1.0 if terminal_loss else 0.0),
            }

            game_signal = sum(terms.values())
            # Strategy weight emphasizes immediate causal evidence and punishes local
            # failure more strongly than game weight, allowing CHANGE_POLICY on good games.
            strategy_signal = (
                5.00 * (1.0 if level_event else 0.0)
                + 2.75 * st.progress_velocity
                + 2.00 * causal_confidence
                + 0.50 * (1.0 if novel else 0.0)
                - 2.50 * stall_ratio
                - 4.00 * repeat_ratio
                - 3.25 * no_progress_ratio
                - 4.50 * (1.0 if terminal_loss else 0.0)
            )

            st.game_weight = _clip(
                GAME_WEIGHT_DECAY * st.game_weight + game_signal,
                -GAME_WEIGHT_LIMIT,
                GAME_WEIGHT_LIMIT,
            )
            st.strategy_weight = _clip(
                STRATEGY_WEIGHT_DECAY * st.strategy_weight + strategy_signal,
                -STRATEGY_WEIGHT_LIMIT,
                STRATEGY_WEIGHT_LIMIT,
            )
            st.combined_weight = 0.55 * st.game_weight + 0.45 * st.strategy_weight

            if level_event or positive_score or positive_reward:
                st.success_protect_until = max(
                    st.success_protect_until,
                    st.move + SUCCESS_PROTECT_ACTIONS,
                )

            st.live_budget = self._budget(st.combined_weight)
            if st.move <= st.success_protect_until:
                st.live_budget = max(st.live_budget, min(LS20_MAX_MOVES, st.success_protect_until + 12))

            protected = st.move <= st.success_protect_until
            if after.get("game_over"):
                st.decision = "TERMINAL"
                st.reason = "environment reported terminal state"
            elif st.move < MIN_OBSERVATION_ACTIONS:
                st.decision = "EXPLORE"
                st.reason = "minimum observation window"
            elif level_event and st.combined_weight >= 3.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "verified level completion"
            elif meaningful_progress and st.combined_weight >= 1.0:
                st.decision = "EXPLOIT"
                st.reason = "verified current-game progress"
            elif protected:
                st.decision = "EXPLOIT_PROTECTED"
                st.reason = "recent success protected exploit window"
            elif st.no_progress_streak >= MAX_NO_PROGRESS_ACTIONS:
                if st.game_weight >= 1.0:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "promising game but current strategy has no progress"
                else:
                    st.decision = "STOP_LOSS"
                    st.reason = "sustained no progress with low game value"
            elif st.stall_streak >= MAX_STALL_ACTIONS:
                if st.game_weight >= 1.0:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "stall detected in a still-promising game"
                elif st.combined_weight < -1.0:
                    st.decision = "STOP_LOSS"
                    st.reason = "hard stall plus negative exploit evidence"
                else:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "stall threshold reached"
            elif st.game_weight >= 1.0 and st.strategy_weight <= -1.0:
                st.decision = "CHANGE_POLICY"
                st.reason = "game weight high while strategy weight is low"
            elif st.combined_weight >= 6.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "very high combined exploit weight"
            elif st.combined_weight >= 3.0:
                st.decision = "EXPLOIT"
                st.reason = "high combined exploit weight"
            elif st.combined_weight >= 1.0:
                st.decision = "CAUTIOUS_EXPLOIT"
                st.reason = "positive combined exploit weight"
            elif st.combined_weight >= -1.0:
                st.decision = "BALANCED"
                st.reason = "mixed current-game evidence"
            else:
                st.decision = "EXPLORE"
                st.reason = "low exploit confidence; seek information"

            st.last_score = float(after_score)
            st.last_levels = int(after_levels)
            if sig:
                st.last_signature = sig
                st.seen_signatures.append(sig)
            st.last_terms = {k: round(v, 6) for k, v in terms.items()}

            event = {
                "game_id": st.game_id,
                "move": st.move,
                "action": action,
                "before_score": before_score,
                "after_score": after_score,
                "score_delta": score_delta,
                "before_levels": before_levels,
                "after_levels": after_levels,
                "level_delta": level_delta,
                "reward": reward,
                "board_changed": bool(board_changed),
                "novel_state": novel,
                "loop_signal": loop_signal,
                "progress_value": progress_value,
                "progress_velocity": st.progress_velocity,
                "causal_confidence": causal_confidence,
                "target_proximity": target_proximity,
                "stall_streak": st.stall_streak,
                "no_progress_streak": st.no_progress_streak,
                "repeat_streak": st.repeat_streak,
                "terms": st.last_terms,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "combined_weight": st.combined_weight,
                "decision": st.decision,
                "reason": st.reason,
                "live_budget": st.live_budget,
                "hard_budget": LS20_MAX_MOVES,
                "success_protect_until": st.success_protect_until,
                "game_over": bool(after.get("game_over")),
                "won": bool(after.get("won")),
                "lost": bool(after.get("lost")),
            }
            with DWE_MOVE_LOG.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(event, sort_keys=True, default=str) + "\n")

            term_text = ",".join(f"{k}={v:+.2f}" for k, v in st.last_terms.items())
            print(
                "DWE POST "
                f"game={st.game_id} move={st.move:03d} action={action} "
                f"score={before_score:.6f}->{after_score:.6f} dscore={score_delta:+.6f} "
                f"levels={before_levels}->{after_levels} dlevel={level_delta:+d} reward={reward:+.4f} "
                f"changed={int(bool(board_changed))} novel={int(novel)} loop={int(loop_signal)} "
                f"progress={progress_value:+.3f} velocity={st.progress_velocity:+.3f} "
                f"target={target_proximity:.3f} stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} next={st.decision} "
                f"budget={st.live_budget}/{LS20_MAX_MOVES} protect_until={st.success_protect_until} "
                f"terms=[{term_text}] reason={st.reason}",
                flush=True,
            )
            return event

    def should_stop(self, game_id):
        with self._lock:
            st = self.state(game_id)
            if st.move >= LS20_MAX_MOVES:
                return True, "hard action ceiling"
            if st.move <= st.success_protect_until:
                return False, "success protected"
            if st.decision == "STOP_LOSS":
                return True, st.reason
            if (
                st.move >= st.live_budget
                and st.no_progress_streak >= MAX_STALL_ACTIONS
                and st.combined_weight < 1.0
            ):
                return True, "dynamic live budget exhausted without exploit evidence"
            return False, "continue"

    def summaries(self):
        with self._lock:
            return [
                {
                    "game_id": st.game_id,
                    "moves": st.move,
                    "score": st.last_score,
                    "levels": st.last_levels,
                    "game_weight": st.game_weight,
                    "strategy_weight": st.strategy_weight,
                    "combined_weight": st.combined_weight,
                    "decision": st.decision,
                    "reason": st.reason,
                    "live_budget": st.live_budget,
                    "stall_streak": st.stall_streak,
                    "no_progress_streak": st.no_progress_streak,
                    "repeat_streak": st.repeat_streak,
                }
                for st in self._states.values()
            ]


DWE_ALLOCATOR = DifferenceWeightedExploitation()
_DWE_ACTION_DEPTH = contextvars.ContextVar("dwe_action_depth", default=0)
_DWE_PATCHED_ACTION_METHODS = []
_DWE_PATCHED_STOP_METHODS = []


def _method_action_score(name, method):
    score = 0
    lname = name.lower()
    if lname in {"step", "act", "action", "execute_action", "perform_action", "take_action", "play_action", "apply_action"}:
        score += 8
    if "action" in lname:
        score += 4
    if any(token in lname for token in ("step", "move", "act", "play")):
        score += 2
    try:
        sig = inspect.signature(method)
        params = {p.lower() for p in sig.parameters}
        if params.intersection({"action", "action_name", "action_spec", "move", "command"}):
            score += 8
    except Exception:
        pass
    try:
        source = inspect.getsource(method).lower()
        if "board_changed" in source or "level_completed" in source or ".step(" in source:
            score += 4
    except Exception:
        pass
    return score


def _wrap_action_method(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_action_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return await original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = await original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            DWE_ALLOCATOR.post(game_id, action, before, after)
            return result
    else:
        def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            DWE_ALLOCATOR.post(game_id, action, before, after)
            return result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_action_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_ACTION_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_gameapi_action_hook(game_apis):
    classes = []
    for api in game_apis:
        if api.__class__ not in classes:
            classes.append(api.__class__)
    installed = []
    for cls in classes:
        candidates = []
        for name in dir(cls):
            if name.startswith("_"):
                continue
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if not callable(method):
                continue
            score = _method_action_score(name, method)
            if score > 0:
                candidates.append((score, name))
        candidates.sort(reverse=True)
        if not candidates:
            raise RuntimeError(
                f"DWE could not identify a real action method on {cls.__module__}.{cls.__name__}; "
                "refusing to run without per-move exploit logging."
            )
        best_score, best_name = candidates[0]
        if best_score < 6:
            raise RuntimeError(
                f"DWE action-boundary confidence too low for {cls.__name__}: {candidates[:8]}"
            )
        if _wrap_action_method(cls, best_name):
            installed.append(f"{cls.__module__}.{cls.__name__}.{best_name}")
        print(
            f"DWE ACTION HOOK class={cls.__module__}.{cls.__name__} "
            f"method={best_name} confidence={best_score} candidates={candidates[:6]}",
            flush=True,
        )
    return installed


def _object_game_id(self, args, kwargs):
    return _extract_game_id(self, *args, *kwargs.values())


def _wrap_should_stop(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_stop_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            original_result = await original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result
    else:
        def wrapped(self, *args, **kwargs):
            original_result = original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_stop_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_STOP_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_should_stop_hooks(game_apis):
    classes = {api.__class__ for api in game_apis}
    # The stop predicate can live on GameAPI, a solver/session class, or a run class.
    for module_name, module in list(sys.modules.items()):
        if not module or not (
            module_name.startswith("taaf") or module_name.startswith("inference")
        ):
            continue
        try:
            values = list(vars(module).values())
        except Exception:
            continue
        for obj in values:
            if inspect.isclass(obj):
                classes.add(obj)

    installed = []
    for cls in classes:
        for name in ("should_stop", "_should_stop"):
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if callable(method) and _wrap_should_stop(cls, name):
                full = f"{cls.__module__}.{cls.__name__}.{name}"
                installed.append(full)
                print(f"DWE STOP HOOK {full}", flush=True)
    if not installed:
        print(
            "DWE STOP HOOK WARNING: no should_stop predicate found; hard ceiling remains active. "
            "Per-move DWE logging and policy weighting are still active.",
            flush=True,
        )
    return installed


def _install_dwe_runtime_hooks(game_apis):
    action_hooks = _install_gameapi_action_hook(game_apis)
    stop_hooks = _install_should_stop_hooks(game_apis)
    if not action_hooks and not _DWE_PATCHED_ACTION_METHODS:
        raise RuntimeError("DWE requires an action-boundary hook; none was installed.")
    print(
        "DWE RUNTIME ACTIVE "
        f"action_hooks={len(action_hooks) or len(_DWE_PATCHED_ACTION_METHODS)} "
        f"stop_hooks={len(stop_hooks) or len(_DWE_PATCHED_STOP_METHODS)} "
        "log_every_move=True",
        flush=True,
    )


print("CLOSED-LOOP ADL ACTIVE", flush=True)
print("DWE ACTIVE: separate game and strategy exploit weights", flush=True)
print("DWE LOGGING: PRE + POST for every real environment action", flush=True)
print("ENVIRONMENT PASSES PER GAME: 1", flush=True)
print("CROSS-GAME MEMORY: DISABLED", flush=True)


In [ ]:
\
# === DWE v3 OVERLAY: HUD/NO-IMPACT + RESULT FEEDBACK ===
import inspect
import json
import types
from collections import deque
from typing import Mapping
DWE_V3_MOVE_LOG = WORKING_DIR / "dwe_v3_move_events.jsonl"
DWE_V3_MOVE_LOG.unlink(missing_ok=True)
_DWE_V3_TRACKERS = {}


def _dwe_grid(value, seen=None):
    if value is None:
        return None
    if seen is None:
        seen = set()
    try:
        ident = id(value)
        if ident in seen:
            return None
        seen.add(ident)
    except Exception:
        pass
    try:
        if hasattr(value, "tolist"):
            value = value.tolist()
    except Exception:
        pass
    if isinstance(value, Mapping):
        for key in ("grid","board","frame","pixels","ascii","data","array","state_matrix","current_frame","after_frame"):
            if key in value:
                got = _dwe_grid(value[key], seen)
                if got is not None:
                    return got
        return None
    if isinstance(value, str):
        lines = [line.rstrip() for line in value.splitlines() if line.strip()]
        rows = []
        for line in lines:
            parts = line.split()
            row = parts if len(parts) > 1 else list(line)
            if row:
                rows.append(tuple(str(x) for x in row))
        if len(rows) >= 2 and len({len(row) for row in rows}) == 1 and len(rows[0]) >= 2:
            return tuple(rows)
        return None
    if isinstance(value, (list, tuple)) and value and all(isinstance(row, (list, tuple)) for row in value):
        widths = {len(row) for row in value}
        if len(widths) == 1 and next(iter(widths), 0) > 0:
            return tuple(tuple(str(x) for x in row) for row in value)
    for attr in ("grid","board","ascii","pixels","array","data","frame","current_frame","after_frame"):
        try:
            child = getattr(value, attr)
        except Exception:
            continue
        if callable(child):
            continue
        got = _dwe_grid(child, seen)
        if got is not None:
            return got
    return None


def _dwe_extract_grid(*objs):
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=3):
            grid = _dwe_grid(candidate)
            if grid is not None:
                return grid
    return None


def _dwe_diff(a, b):
    if a is None or b is None or len(a) != len(b):
        return None
    if any(len(x) != len(y) for x, y in zip(a, b)):
        return None
    return {(r,c) for r,(ra,rb) in enumerate(zip(a,b)) for c,(x,y) in enumerate(zip(ra,rb)) if x != y}


def _dwe_masked_signature(grid, rows=(), cols=()):
    if grid is None:
        return None
    rows, cols = set(rows), set(cols)
    payload = [["." if r in rows or c in cols else str(v) for c,v in enumerate(row)] for r,row in enumerate(grid)]
    raw = json.dumps(payload, separators=(",",":"), ensure_ascii=False)
    return hashlib.sha1(raw.encode("utf-8", "replace")).hexdigest()[:16]


def _dwe_tracker(game_id):
    key = str(game_id or "unknown")
    if key not in _DWE_V3_TRACKERS:
        _DWE_V3_TRACKERS[key] = {
            "history": deque(maxlen=NO_IMPACT_BAND_WINDOW),
            "shape": None,
            "band_rows": (),
            "band_cols": (),
            "no_impact_streak": 0,
            "no_impact_total": 0,
            "last_source": "none",
        }
    return _DWE_V3_TRACKERS[key]


def _dwe_band(t):
    history = list(t["history"])
    if len(history) < NO_IMPACT_BAND_WARMUP:
        return (), ()
    denom = float(len(history))
    rc, cc = {}, {}
    for rows, cols in history:
        for r in rows:
            rc[r] = rc.get(r, 0) + 1
        for c in cols:
            cc[c] = cc.get(c, 0) + 1
    return (
        tuple(sorted(r for r,n in rc.items() if n/denom >= NO_IMPACT_BAND_THRESHOLD)),
        tuple(sorted(c for c,n in cc.items() if n/denom >= NO_IMPACT_BAND_THRESHOLD)),
    )


def _dwe_classify(game_id, before_grid, after_grid, meaningful_progress):
    t = _dwe_tracker(game_id)
    changed = _dwe_diff(before_grid, after_grid)
    if changed is None:
        t["no_impact_streak"] = 0
        t["last_source"] = "unavailable"
        return False, "unavailable", t["band_rows"], t["band_cols"], None
    shape = (len(after_grid or ()), len(after_grid[0]) if after_grid else 0)
    if t["shape"] is not None and shape != t["shape"]:
        t["history"].clear(); t["band_rows"] = (); t["band_cols"] = (); t["no_impact_streak"] = 0
    t["shape"] = shape
    prior_rows, prior_cols = _dwe_band(t)
    no_impact = bool(changed and not meaningful_progress and (prior_rows or prior_cols) and all(r in prior_rows or c in prior_cols for r,c in changed))
    t["history"].append((tuple(sorted({r for r,_ in changed})), tuple(sorted({c for _,c in changed}))))
    t["band_rows"], t["band_cols"] = _dwe_band(t)
    if no_impact:
        t["no_impact_streak"] += 1; t["no_impact_total"] += 1; t["last_source"] = "band"
    else:
        t["no_impact_streak"] = 0; t["last_source"] = "exact-static" if not changed else "band-learning"
    canonical = _dwe_masked_signature(after_grid, t["band_rows"], t["band_cols"])
    return no_impact, t["last_source"], t["band_rows"], t["band_cols"], canonical


_DWE_V2_SNAPSHOT = _snapshot

def _snapshot(api, result=None):
    snap = _DWE_V2_SNAPSHOT(api, result)
    snap["grid"] = _dwe_extract_grid(result, api)
    return snap


_DWE_V2_PRE = DWE_ALLOCATOR.pre
_DWE_V2_POST = DWE_ALLOCATOR.post
_DWE_V2_SUMMARIES = DWE_ALLOCATOR.summaries


def _dwe_v3_pre(self, game_id, action):
    out = _DWE_V2_PRE(game_id, action)
    t = _dwe_tracker(game_id)
    print(
        "DWE PRE+ "
        f"game={game_id} action={action} no_impact={t['no_impact_streak']}/{MAX_NO_IMPACT_ACTIONS} "
        f"hud_rows={list(t['band_rows'])} hud_cols={list(t['band_cols'])} seed={CONTROL_SEED}",
        flush=True,
    )
    return out


def _dwe_v3_post(self, game_id, action, before, after):
    bscore = _num(before.get("score"), 0.0) or 0.0
    ascore = _num(after.get("score"), bscore)
    reward = _num(after.get("reward"), 0.0) or 0.0
    level_event = bool(after.get("level_completed"))
    meaningful = bool(level_event or (ascore is not None and ascore > bscore + 1e-9) or reward > 1e-9)
    no_impact, source, rows, cols, canonical = _dwe_classify(game_id, before.get("grid"), after.get("grid"), meaningful)
    adjusted = dict(after)
    if canonical:
        adjusted["signature"] = canonical
    if no_impact:
        adjusted["board_changed"] = False
    event = _DWE_V2_POST(game_id, action, before, adjusted)
    st = self.state(game_id)
    t = _dwe_tracker(game_id)
    ratio = _clip(t["no_impact_streak"] / max(MAX_NO_IMPACT_ACTIONS, 1), 0.0, 1.0)
    no_impact_term = EXPLOIT_WEIGHTS["no_impact"] * ratio
    if no_impact_term:
        st.game_weight = _clip(st.game_weight + 0.20 * no_impact_term, -GAME_WEIGHT_LIMIT, GAME_WEIGHT_LIMIT)
        st.strategy_weight = _clip(st.strategy_weight + no_impact_term, -STRATEGY_WEIGHT_LIMIT, STRATEGY_WEIGHT_LIMIT)
        st.combined_weight = 0.55 * st.game_weight + 0.45 * st.strategy_weight
        st.live_budget = self._budget(st.combined_weight)
    protected = st.move <= st.success_protect_until
    if not protected and t["no_impact_streak"] >= MAX_NO_IMPACT_ACTIONS:
        if st.game_weight >= 0.5:
            st.decision = "CHANGE_POLICY"; st.reason = "repeated housekeeping-only/no-impact actions"
        elif st.no_progress_streak >= MAX_STALL_ACTIONS:
            st.decision = "STOP_LOSS"; st.reason = "no-impact streak plus low game value"
        else:
            st.decision = "CHANGE_POLICY"; st.reason = "no-impact threshold reached"
    event.update({
        "no_impact": bool(no_impact), "no_impact_source": source,
        "no_impact_streak": t["no_impact_streak"], "no_impact_total": t["no_impact_total"],
        "hud_band_rows": list(rows), "hud_band_cols": list(cols), "no_impact_term": no_impact_term,
        "game_weight": st.game_weight, "strategy_weight": st.strategy_weight,
        "combined_weight": st.combined_weight, "decision": st.decision, "reason": st.reason,
        "live_budget": st.live_budget, "control_seed": CONTROL_SEED, "model_id": ANALYZER_MODEL_ID,
    })
    with DWE_V3_MOVE_LOG.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(event, sort_keys=True, default=str) + "\n")
    print(
        "DWE POST+ "
        f"game={game_id} move={st.move:03d} action={action} no_impact={int(no_impact)} source={source} "
        f"no_impact_streak={t['no_impact_streak']}/{MAX_NO_IMPACT_ACTIONS} hud_rows={list(rows)} hud_cols={list(cols)} "
        f"no_impact_term={no_impact_term:+.3f} gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
        f"combined={st.combined_weight:+.3f} next={st.decision} budget={st.live_budget}/{LS20_MAX_MOVES} reason={st.reason}",
        flush=True,
    )
    return event


def _dwe_v3_summaries(self):
    items = _DWE_V2_SUMMARIES()
    for item in items:
        t = _dwe_tracker(item["game_id"])
        item.update({
            "no_impact_streak": t["no_impact_streak"], "no_impact_total": t["no_impact_total"],
            "last_no_impact_source": t["last_source"], "hud_band_rows": list(t["band_rows"]), "hud_band_cols": list(t["band_cols"]),
        })
    return items


DWE_ALLOCATOR.pre = types.MethodType(_dwe_v3_pre, DWE_ALLOCATOR)
DWE_ALLOCATOR.post = types.MethodType(_dwe_v3_post, DWE_ALLOCATOR)
DWE_ALLOCATOR.summaries = types.MethodType(_dwe_v3_summaries, DWE_ALLOCATOR)


def _dwe_annotate_result(result, event):
    if result is None:
        return
    patch = {
        "dwe_decision": event.get("decision"), "dwe_game_weight": event.get("game_weight"),
        "dwe_strategy_weight": event.get("strategy_weight"), "dwe_combined_weight": event.get("combined_weight"),
        "dwe_no_impact": event.get("no_impact"), "dwe_no_impact_source": event.get("no_impact_source"),
        "dwe_live_budget": event.get("live_budget"), "dwe_reason": event.get("reason"),
    }
    if isinstance(result, dict):
        result.update(patch); return
    for key, value in patch.items():
        try:
            setattr(result, key, value)
        except Exception:
            pass


def _wrap_action_method(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_action_wrapper", False):
        return False
    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return await original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self); action = _extract_action(args, kwargs); before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = await original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            event = DWE_ALLOCATOR.post(game_id, action, before, _snapshot(self, result))
            _dwe_annotate_result(result, event)
            return result
    else:
        def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self); action = _extract_action(args, kwargs); before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            event = DWE_ALLOCATOR.post(game_id, action, before, _snapshot(self, result))
            _dwe_annotate_result(result, event)
            return result
    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_action_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_ACTION_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


print(f"DWE v3 OVERLAY ACTIVE seed={CONTROL_SEED} model={ANALYZER_MODEL_ID} no_impact=statistical-band result_feedback=on", flush=True)


## 8. Run exactly one real environment trajectory per game

Competition and local modes share the same policy. Competition mode discovers games
from the official gateway. Local mode uses the mounted public `environment_files`.


In [ ]:
# === ONE-ENVIRONMENT COMPETITION/LOCAL EXECUTION ===
import re
from urllib.request import urlopen

EXPECTED_LOCAL_GAME_COUNT = 25


def _game_key(value):
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive game key from {value!r}")
    return match.group(1)


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_games():
    import arc_agi
    import taaf.game_api

    root = Path(os.environ["ARC_AGI3_ENVIRONMENTS_DIR"])
    if not root.is_dir():
        raise FileNotFoundError(f"Local environment root missing: {root}")

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Expected {EXPECTED_LOCAL_GAME_COUNT} local games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Competition gateway did not become ready: {last_error}")


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    game_apis = _competition_games()
    print(
        f"OFFICIAL COMPETITION MODE: {len(game_apis)} games, "
        "one environment trajectory per game",
        flush=True,
    )
else:
    game_apis = _offline_games()
    print(
        f"LOCAL MODE: {len(game_apis)} games, one environment trajectory per game",
        flush=True,
    )

RUN_GAME_COUNT = len(game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No ARC-AGI-3 games discovered.")

# Install deterministic per-move exploit logging and stop-loss hooks only after
# the actual GameAPI class(es) have been constructed. The notebook refuses to
# start the benchmark if it cannot identify the action boundary confidently.
_install_dwe_runtime_hooks(game_apis)

# One pass only. No hidden-game restart and no environment best-of-two.
bm.games = game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY

# Runtime budget: reserve setup/teardown time, then divide the remaining time
# across the one legal environment pass.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 70 * 60
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY / RUN_GAME_COUNT,
)
# Keep the per-game runtime aligned with the ls20 move budget target.
RUN_PER_GAME_SECONDS = min(
    1500.0,
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else runtime_safe_per_game,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS

run_manifest = {
    "ls20_max_moves": LS20_MAX_MOVES,
    "schema": "adldb.arc3.dwe.v2",
    "dwe_enabled": True,
    "dwe_log_every_move": True,
    "dwe_weights": EXPLOIT_WEIGHTS,
    "dwe_budget_tiers": [list(x) for x in DWE_BUDGET_TIERS],
    "success_protect_actions": SUCCESS_PROTECT_ACTIONS,
    "min_observation_actions": MIN_OBSERVATION_ACTIONS,
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": RUN_GAME_COUNT,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "concurrency": TARGET_CONCURRENCY,
    "per_game_seconds": RUN_PER_GAME_SECONDS,
    "strict_no_prior": True,
    "second_environment_pass": False,
    "target_score_games": TARGET_SCORE_GAMES,
    "target_min_game_score": TARGET_MIN_GAME_SCORE,
    "max_stall_actions": MAX_STALL_ACTIONS,
    "max_no_progress_actions": MAX_NO_PROGRESS_ACTIONS,
}

(WORKING_DIR / "dual_path_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(
    "DUAL-PATH RUN START "
    f"games={RUN_GAME_COUNT} concurrency={TARGET_CONCURRENCY} "
    f"per_game_seconds={RUN_PER_GAME_SECONDS:.1f} "
    f"hard_action_ceiling={LS20_MAX_MOVES} dwe=on log_every_move=on",
    flush=True,
)

try:
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=bool(TRUE_SUBMISSION),
    )
finally:
    # Keep the source bundle's own teardown behavior.
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if len(getattr(bm, "game_runs", []) or []) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Expected {RUN_GAME_COUNT} completed game runs; "
        f"found {len(getattr(bm, 'game_runs', []) or [])}"
    )

for run in bm.game_runs:
    print(
        "ADLDB-DWE SCORE "
        f"game={_game_key(run.game_id)} "
        f"score={_run_score(run):.6f} "
        f"levels={_run_levels(run)} "
        f"actions={_run_actions(run)}",
        flush=True,
    )


## 9. Write and validate `submission.parquet`


In [ ]:
# === VALIDATED COMPETITION ARTIFACT ===
import pandas as pd

rows = [
    {
        "row_id": f"{run.game_id}_0",
        "game_id": str(run.game_id),
        "end_of_game": _won(run),
        "score": _run_score(run),
    }
    for run in bm.game_runs
]

if len(rows) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Submission requires {RUN_GAME_COUNT} rows; found {len(rows)}"
    )

submission = pd.DataFrame(
    rows,
    columns=["row_id", "game_id", "end_of_game", "score"],
)

if submission["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
    raise RuntimeError("Submission contains duplicate game IDs.")
if submission["score"].isna().any():
    raise RuntimeError("Submission contains missing scores.")

SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
submission.to_parquet(SUBMISSION_PATH, index=False)

check = pd.read_parquet(SUBMISSION_PATH)
if list(check.columns) != ["row_id", "game_id", "end_of_game", "score"]:
    raise RuntimeError(f"Invalid submission columns: {list(check.columns)}")
if len(check) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Written submission row count mismatch: {len(check)} != {RUN_GAME_COUNT}"
    )

print(
    "SUBMISSION READY "
    f"path={SUBMISSION_PATH} rows={len(check)} "
    f"score_sum={float(check['score'].sum()):.6f}",
    flush=True,
)


## 10. Final ADLDB/DWE run summary

Summarize scored outcomes and the final exploit state of every game. The output keeps the score metrics needed for the next ADL comparison and writes a compact per-game DWE summary artifact.


In [ ]:
# === FINAL ADLDB / DWE SUMMARY ===
import statistics

runs = list(getattr(bm, "game_runs", []) or [])
scores = [_run_score(run) for run in runs]
levels = [_run_levels(run) for run in runs]
actions = [_run_actions(run) for run in runs]
positive = sum(score > 0 for score in scores)
at_target = sum(score >= TARGET_MIN_GAME_SCORE for score in scores)

print(
    f"ADLDB SUMMARY model={ANALYZER_MODEL_ID} seed={CONTROL_SEED} "
    f"games={len(runs)} "
    f"mean_score={(sum(scores) / len(scores) if scores else 0.0):.6f} "
    f"score_sum={sum(scores):.6f} "
    f"positive_games={positive} "
    f"target_games={at_target}/{TARGET_SCORE_GAMES} "
    f"levels={sum(levels)} "
    f"actions={sum(actions)}",
    flush=True,
)

summaries = DWE_ALLOCATOR.summaries()
with DWE_SUMMARY_LOG.open("w", encoding="utf-8") as fh:
    for item in sorted(summaries, key=lambda x: x["game_id"]):
        fh.write(json.dumps(item, sort_keys=True) + "\n")
        print(
            "DWE GAME SUMMARY "
            f"game={item['game_id']} moves={item['moves']} score={item['score']:.6f} "
            f"levels={item['levels']} gameW={item['game_weight']:+.3f} "
            f"strategyW={item['strategy_weight']:+.3f} combined={item['combined_weight']:+.3f} "
            f"decision={item['decision']} live_budget={item['live_budget']} "
            f"stall={item['stall_streak']} no_progress={item['no_progress_streak']} "
            f"no_impact={item.get('no_impact_streak', 0)} total_no_impact={item.get('no_impact_total', 0)} "
            f"repeat={item['repeat_streak']} reason={item['reason']}",
            flush=True,
        )

print(f"DWE BASE MOVE LOG: {DWE_MOVE_LOG}", flush=True)
print(f"DWE v3 MOVE LOG: {DWE_V3_MOVE_LOG}", flush=True)
print(f"DWE SUMMARY LOG: {DWE_SUMMARY_LOG}", flush=True)


## 11. Per-move exploit/ADL audit

Verify that the deterministic DWE action-boundary logger produced one unique post-action exploit record for every recorded game action. In local validation mode, incomplete coverage is a hard failure. In official competition reruns it is surfaced prominently without destroying an otherwise valid submission artifact.


In [ ]:
# === PER-MOVE DWE / POST-MOVE ADL AUDIT ===
from collections import Counter

records = []
if DWE_V3_MOVE_LOG.exists():
    for raw in DWE_V3_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        try:
            records.append(json.loads(raw))
        except json.JSONDecodeError:
            print("DWE AUDIT malformed line:", raw[:240], flush=True)

unique_move_keys = {
    (str(item.get("game_id", "unknown")), int(item.get("move", -1)))
    for item in records
    if item.get("move") is not None
}
recorded_actions = sum(_run_actions(run) for run in (getattr(bm, "game_runs", []) or []))
logged_moves = len(unique_move_keys)
coverage = logged_moves / recorded_actions if recorded_actions else 1.0

modes = Counter(str(item.get("decision", "unknown")) for item in records)
stop_loss = modes.get("STOP_LOSS", 0)
change_policy = modes.get("CHANGE_POLICY", 0)
exploit_moves = sum(
    count for mode, count in modes.items()
    if "EXPLOIT" in mode
)
no_impact_moves = sum(bool(item.get("no_impact")) for item in records)

print(
    "DWE MOVE AUDIT "
    f"recorded_actions={recorded_actions} "
    f"unique_logged_moves={logged_moves} "
    f"coverage={coverage:.3f} "
    f"exploit_updates={exploit_moves} "
    f"change_policy_updates={change_policy} "
    f"stop_loss_updates={stop_loss} "
    f"no_impact_moves={no_impact_moves} "
    f"modes={dict(sorted(modes.items()))}",
    flush=True,
)

# Per-game coverage makes any missing trace immediately visible in notebook logs.
logged_by_game = Counter(str(item.get("game_id", "unknown")) for item in records)
for run in getattr(bm, "game_runs", []) or []:
    gid = _game_key(run.game_id)
    # Match either exact full id or normalized game key.
    logged = sum(
        count for key, count in logged_by_game.items()
        if _game_key(key) == gid
    )
    expected = _run_actions(run)
    game_cov = logged / expected if expected else 1.0
    print(
        "DWE GAME AUDIT "
        f"game={gid} actions={expected} logged={logged} coverage={game_cov:.3f}",
        flush=True,
    )

if coverage < 0.999999:
    message = (
        "DWE per-move exploit logging coverage is incomplete: "
        f"{logged_moves}/{recorded_actions} ({coverage:.3%})."
    )
    if DWE_STRICT_LOG_COVERAGE and not TRUE_SUBMISSION:
        raise RuntimeError(message)
    print("WARNING:", message, flush=True)
else:
    print("DWE AUDIT PASS: exploit logic logged for every recorded move", flush=True)
